In [2]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q fastapi uvicorn pydantic python-multipart pillow opencv-python-headless
!pip install -q torchxrayvision scikit-image pyngrok

Mounted at /content/drive


In [3]:
%%writefile app.py
import base64
import io
import os
from typing import Optional, Dict, Any

import cv2
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications.resnet50 import preprocess_input
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image

tf.config.set_visible_devices([], "GPU")

app = FastAPI(title="InnoTech X-ray Service", version="1.0.0-colab")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

GRADCAM_TARGET_LAYER = os.getenv("GRADCAM_TARGET_LAYER", "conv4_block6_out")

MODEL_CANDIDATES = [
    os.getenv("MODEL_PATH", ""),
    "/content/drive/MyDrive/InnoTech_Dataset/ResNet50_merged_fixed.keras",
    "/content/drive/MyDrive/InnoTech_Dataset/Models_CNN/ResNet50_merged_fixed.keras",
    "./ResNet50_merged_fixed.keras",
]


xray_model = None
base_grad_model = None
head_layers = []
xray_model_error: Optional[str] = None


def _find_layer_recursive(model_obj, target_name: str):
    for layer in getattr(model_obj, "layers", []):
        if layer.name == target_name:
            return layer
        if hasattr(layer, "layers"):
            found = _find_layer_recursive(layer, target_name)
            if found is not None:
                return found
    return None


def _load_classifier():
    global xray_model, base_grad_model, head_layers, xray_model_error
    path = next((p for p in MODEL_CANDIDATES if p and os.path.isfile(p)), None)
    if not path:
        xray_model_error = "ResNet50_merged_fixed.keras not found (is Google Drive mounted?)."
        print(f"[startup] {xray_model_error}")
        return
    try:
        loaded = keras.models.load_model(path, compile=False)
        backbone, head = None, []
        for idx, lyr in enumerate(loaded.layers):
            if isinstance(lyr, keras.Model) and _find_layer_recursive(lyr, GRADCAM_TARGET_LAYER) is not None:
                backbone, head = lyr, loaded.layers[idx + 1:]
                break
        if backbone is None:
            backbone, head = loaded, []
        target = _find_layer_recursive(backbone, GRADCAM_TARGET_LAYER)
        if target is None:
            raise ValueError(f"Layer '{GRADCAM_TARGET_LAYER}' not found in the model.")
        base_grad_model = keras.Model(inputs=backbone.inputs, outputs=[target.output, backbone.output])
        head_layers = head
        xray_model = loaded
        print(f"[startup] Classifier loaded from {path}")
    except Exception as e:
        xray_model, base_grad_model, head_layers = None, None, []
        xray_model_error = f"{type(e).__name__}: {e}"
        print(f"[startup] Classifier NOT loaded: {xray_model_error}")


_load_classifier()


seg_model = None
seg_device = "cpu"
xrv = None
torch = None
try:
    import torch as _torch
    import torchvision.transforms as _transforms
    import torchxrayvision as _xrv

    torch, xrv, transforms = _torch, _xrv, _transforms
    seg_device = "cuda" if torch.cuda.is_available() else "cpu"
    seg_model = xrv.baseline_models.chestx_det.PSPNet().to(seg_device).eval()
    print("[startup] Lung segmentation model loaded.")
except Exception as e:
    print(f"[startup] Segmentation disabled: {type(e).__name__}: {e}")


def _rgb_to_b64(rgb: np.ndarray) -> str:
    ok, buf = cv2.imencode(".png", cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
    if not ok:
        raise RuntimeError("PNG encoding failed.")
    return base64.b64encode(buf.tobytes()).decode("utf-8")


def compute_lung_segmentation(pil_img_rgb: Image.Image):
    gray = np.array(pil_img_rgb.convert("L"), dtype=np.float32)
    img_norm = xrv.datasets.normalize(gray, 255)[None, ...]
    img_resized = transforms.Compose([xrv.datasets.XRayResizer(512)])(img_norm)
    tensor = torch.from_numpy(img_resized).unsqueeze(0).to(seg_device)
    with torch.no_grad():
        pred = seg_model(tensor)
    li = seg_model.targets.index("Left Lung")
    ri = seg_model.targets.index("Right Lung")
    prob = torch.sigmoid(pred[0, li]) + torch.sigmoid(pred[0, ri])
    mask_512 = (prob.cpu().numpy() > 0.5).astype(np.uint8) * 255
    mask_224 = cv2.resize(mask_512, (224, 224), interpolation=cv2.INTER_NEAREST)
    orig_224 = np.array(pil_img_rgb.resize((224, 224)), dtype=np.uint8)
    contours, _ = cv2.findContours(mask_224, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    overlay = orig_224.copy()
    cv2.drawContours(overlay, contours, -1, (0, 255, 0), 2)
    return _rgb_to_b64(np.stack([mask_224] * 3, axis=-1)), _rgb_to_b64(overlay)


def compute_gradcam(pil_rgb: Image.Image) -> Dict[str, Any]:
    pil_224 = pil_rgb.resize((224, 224), Image.Resampling.BILINEAR)
    orig_rgb = np.array(pil_224, dtype=np.uint8)
    x_in = tf.cast(preprocess_input(np.expand_dims(orig_rgb.astype(np.float32), 0)), tf.float32)

    with tf.GradientTape() as tape:
        conv_outputs, backbone_out = base_grad_model(x_in, training=False)
        tape.watch(conv_outputs)
        x = backbone_out
        for layer in head_layers:
            x = layer(x, training=False)
        score = x[:, 0] if len(x.shape) > 1 else x
        loss = tf.reduce_mean(score)

    grads = tape.gradient(loss, conv_outputs)
    raw = float(score.numpy()[0])
    prob = raw if 0.0 <= raw <= 1.0 else float(1.0 / (1.0 + np.exp(-raw)))

    if grads is not None:
        pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
        heat = tf.maximum(tf.reduce_sum(tf.multiply(pooled, conv_outputs[0]), axis=-1), 0)
        mx = float(tf.reduce_max(heat).numpy())
        heat_np = (heat / mx).numpy() if mx > 1e-8 else heat.numpy()
    else:
        heat_np = np.zeros((7, 7), dtype=np.float32)

    heat_224 = cv2.resize(heat_np, (224, 224), interpolation=cv2.INTER_LINEAR)
    heat_u8 = np.uint8(255 * np.clip(heat_224, 0.0, 1.0))
    heat_rgb = cv2.cvtColor(cv2.applyColorMap(heat_u8, cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(orig_rgb, 0.65, heat_rgb, 0.35, 0)

    abnormal = prob >= 0.5
    return {
        "prediction": "ABNORMAL / FINDING" if abnormal else "NORMAL",
        "visual_finding": (
            "Abnormal Opacity / Attention Detected" if abnormal else "Unremarkable / Clear Lung Fields"
        ),
        "confidence": round(prob if abnormal else 1.0 - prob, 4),
        "raw_score": round(prob, 4),
        "target_layer": GRADCAM_TARGET_LAYER,
        "gradcam_base64": _rgb_to_b64(overlay),
        "heatmap_base64": _rgb_to_b64(heat_rgb),
        "overlay_base64": _rgb_to_b64(overlay),
    }


@app.get("/")
def root():
    return {"status": "online", "service": "InnoTech X-ray Service", "docs": "/docs"}


@app.get("/health")
def health():
    return {
        "status": "ok",
        "gradcam_ready": base_grad_model is not None,
        "segmentation_ready": seg_model is not None,
        "model_error": xray_model_error,
    }


@app.post("/predict")
def predict_xray(file: UploadFile = File(...)):
    if base_grad_model is None:
        raise HTTPException(status_code=503, detail=f"X-ray model not loaded: {xray_model_error}")
    try:
        raw_pil = Image.open(io.BytesIO(file.file.read())).convert("RGB")
    except Exception:
        raise HTTPException(status_code=400, detail="Corrupted or unreadable image file.")

    try:
        result = compute_gradcam(raw_pil)
    except Exception as e:
        print(f"[/predict] Grad-CAM failed: {type(e).__name__}: {e}")
        raise HTTPException(status_code=500, detail=f"Grad-CAM error: {type(e).__name__}: {e}")

    result["segmentation_mask_base64"] = None
    result["segmentation_overlay_base64"] = None
    if seg_model is not None:
        try:
            mask_b64, overlay_b64 = compute_lung_segmentation(raw_pil)
            result["segmentation_mask_base64"] = mask_b64
            result["segmentation_overlay_base64"] = overlay_b64
        except Exception as e:
            print(f"[/predict] Segmentation failed: {type(e).__name__}: {e}")

    result["status"] = "success"
    result["disclaimer"] = "AI visual decision-support output only. Not a definitive clinical diagnosis."
    return result

Writing app.py


In [4]:
!pkill -f "[u]vicorn app:app"
!nohup uvicorn app:app --host 0.0.0.0 --port 8000 > server.log 2>&1 &

import time, requests
ready = False
for _ in range(120):
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=2).ok:
            ready = True
            break
    except Exception:
        pass
    time.sleep(5)

print("SERVER READY" if ready else "SERVER FAILED - Check server.log")
!tail -n 30 server.log

SERVER READY
I0000 00:00:1795878703.954487     718 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[startup] Classifier loaded from /content/drive/MyDrive/InnoTech_Dataset/Models_CNN/ResNet50_merged_fixed.keras
If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/pspnet_chestxray_best_model_4.pth -O /root/.torchxrayvision/models_data/pspnet_chestxray_best_model_4.pth`
[██████████████████████████████████████████████████]INFO:     Started server process [718]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)

[startup] Lung segmentation model loaded.
INFO:     127.0.0.1:51454 - "GET /health HTTP/1.1" 200 OK


In [5]:
from pyngrok import ngrok
from google.colab import userdata

ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))

STATIC_DOMAIN = "gaffe-arrive-t-shirt.ngrok-free.dev"
ngrok.kill()

public_url = ngrok.connect(8000, url=f"https://{STATIC_DOMAIN}")
print("Static URL:", public_url.public_url)

Static URL: https://gaffe-arrive-t-shirt.ngrok-free.dev


In [6]:
import requests
print(requests.get(f"{public_url.public_url}/health", timeout=60).json())

{'status': 'ok', 'gradcam_ready': True, 'segmentation_ready': True, 'model_error': None}
